# 📊 Model Evaluation — Retail Demand Forecasting

Detailed evaluation: residual analysis, feature importance, prediction distribution.

**Reads from:** `gold_ml_features`, saved model

**Writes to:** `gold_ml_feature_importance`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, abs as spark_abs, avg
from pyspark.ml.feature import VectorAssembler, StringIndexer
from synapse.ml.lightgbm import LightGBMRegressionModel

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
model = LightGBMRegressionModel.load('Files/models/demand_forecasting_lgbm')
print('Model loaded')

In [ ]:
# Prepare test data (same pipeline as training)
features_df = spark.read.table('gold_ml_features')

numeric_features = [
    'transaction_count', 'avg_discount', 'avg_price', 'daily_revenue',
    'day_of_week', 'day_of_month', 'month', 'week_of_year', 'is_weekend',
    'demand_lag_1', 'demand_lag_7', 'revenue_lag_1',
    'unit_cost', 'margin_pct',
]

cat_cols = ['category', 'subcategory', 'region', 'store_format']
indexed_df = features_df
cat_idx_cols = []
for c in cat_cols:
    idx_col = f'{c}_idx'
    indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid='keep')
    indexed_df = indexer.fit(indexed_df).transform(indexed_df)
    cat_idx_cols.append(idx_col)

all_features = numeric_features + cat_idx_cols
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='skip')
model_df = assembler.transform(indexed_df)

_, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
predictions = model.transform(test_df)
print(f'Test predictions: {predictions.count():,} rows')

In [ ]:
# Residual analysis
residuals = (
    predictions
    .withColumn('residual', col('daily_quantity') - col('prediction'))
    .withColumn('abs_residual', spark_abs(col('residual')))
    .withColumn('pct_error',
        spark_abs(col('residual')) / col('daily_quantity') * 100
    )
)

print('=== Residual Statistics ===')
residuals.select('residual', 'abs_residual', 'pct_error').describe().show()

mape = residuals.filter(col('daily_quantity') > 0).agg(avg('pct_error')).collect()[0][0]
print(f'MAPE: {mape:.2f}%')

In [ ]:
# Feature importance
import pandas as pd

try:
    importances = model.getFeatureImportances(importance_type='gain')
    fi_df = pd.DataFrame({
        'feature': all_features,
        'importance': importances[:len(all_features)]
    }).sort_values('importance', ascending=False)
    
    print('=== Top 10 Features by Gain ===')
    for _, row in fi_df.head(10).iterrows():
        print(f"  {row['feature']:30s} {row['importance']:.4f}")
    
    fi_spark = spark.createDataFrame(fi_df).withColumn('model_timestamp', current_timestamp())
    fi_spark.write.mode('overwrite').format('delta').saveAsTable('gold_ml_feature_importance')
    print('\nFeature importance saved')
except Exception as e:
    print(f'Could not extract feature importance: {e}')
    fi_data = [(f, 0.0) for f in all_features]
    fi_spark = spark.createDataFrame(fi_data, ['feature', 'importance']).withColumn('model_timestamp', current_timestamp())
    fi_spark.write.mode('overwrite').format('delta').saveAsTable('gold_ml_feature_importance')
    print('Placeholder feature importance saved')

In [ ]:
spark.sql('OPTIMIZE gold_ml_feature_importance')
print('Evaluation complete')